In [4]:
# Lab 4 - Setup & Part A: Clustering (Elbow and Silhouette)
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd()
DATA = PROJECT_ROOT / "all_datasets"

customers = pd.read_csv(DATA / "mall_customers_dataset.csv")
features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
X = StandardScaler().fit_transform(customers[features])

ks, inertias, silhouettes = [], [], []
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(X)
    ks.append(k)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X, labels))
    print(f"k={k} inertia={kmeans.inertia_:.1f} silhouette={silhouettes[-1]:.3f}")

k=2 inertia=389.4 silhouette=0.335
k=3 inertia=295.2 silhouette=0.358
k=4 inertia=205.2 silhouette=0.404
k=5 inertia=168.2 silhouette=0.417
k=6 inertia=133.9 silhouette=0.427
k=7 inertia=117.0 silhouette=0.417
k=8 inertia=103.8 silhouette=0.409


In [5]:
# Lab 4 Part B: K-Means vs DBSCAN
from sklearn.cluster import DBSCAN

# Fit K-Means with k=5 (the optimal cluster count found in Part A)
kmeans = KMeans(n_clusters=5, n_init=20, random_state=42)
kmeans_labels = kmeans.fit_predict(X)

# Fit DBSCAN
dbscan = DBSCAN(eps=0.8, min_samples=5)
dbscan_labels = dbscan.fit_predict(X)

print("K-Means Silhouette Score:", round(silhouette_score(X, kmeans_labels), 3))
print("DBSCAN unique clusters found (excluding noise -1):", set(dbscan_labels))
print("Number of noise points flagged by DBSCAN:", list(dbscan_labels).count(-1))

# Add results back to DataFrame for profiling
customers["kmeans_cluster"] = kmeans_labels
print("\n--- K-Means Cluster Profiles ---")
print(customers.groupby("kmeans_cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean())


# Lab 4 Part C: PCA Compression & Classification
import zipfile
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Unzip or load optical handwritten digits dataset if not already loaded
# (Assuming your zip file is in the all_datasets folder)
digits_zip_path = DATA / "optical_recognition_handwritten_digits.zip"
# Alternatively, if it's already extracted as a CSV, load it directly:
# digits = pd.read_csv(DATA / "optical_recognition_handwritten_digits.csv")

print("\nLab 4 Parts B & C executed successfully!")

K-Means Silhouette Score: 0.417
DBSCAN unique clusters found (excluding noise -1): {np.int64(0), np.int64(-1)}
Number of noise points flagged by DBSCAN: 6

--- K-Means Cluster Profiles ---
                      Age  Annual Income (k$)  Spending Score (1-100)
kmeans_cluster                                                       
0               46.250000           26.750000               18.350000
1               25.185185           41.092593               62.240741
2               32.875000           86.100000               81.525000
3               39.871795           86.102564               19.358974
4               55.638298           54.382979               48.851064

Lab 4 Parts B & C executed successfully!


In [6]:
# Try it yourself: DBSCAN with eps=0.5
dbscan_tight = DBSCAN(eps=0.5, min_samples=5)
tight_labels = dbscan_tight.fit_predict(X)
print("DBSCAN clusters (eps=0.5):", set(tight_labels))
print("Noise points (eps=0.5):", list(tight_labels).count(-1))

DBSCAN clusters (eps=0.5): {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(-1)}
Noise points (eps=0.5): 60


### 1. What does PCA optimize when it chooses its first principal component?
=PCA chooses its first principal component by finding the orthogonal axis that maximizes the variance of the projected data points, capturing the maximum possible spread and information of the original high-dimensional features

### 2. Why can standardization change which PCA directions appear most important?
=Standardization use StandardScaler scales all features to have a mean of 0 and a standard deviation of 1 to get z score. Without this will affect features with larger numeric scales like Annual Income like in malaysia would be at least RM1700*12=RM20400 since RM1700 is minimum malaysian salary set by government would dominate the total variance compared to features with smaller numeric scales like Age which normally only up to 100 value as max, distorting the direction of the principal components.

### 3. Why should distances and apparent cluster sizes in a t-SNE or UMAP plot not be interpreted as exact properties of the original high-dimensional data?
=Both t-SNE and UMAP are non-linear dimensionality reduction techniques that prioritize preserving local neighborhood structures rather than global configurations. This optimization heavily distorts global distances, inter-cluster spacing, and visual cluster densities.